# Aula 5, GPT

Notebook executável que acompanha a aula [05-gpt.md](../../lessons/modulo-06-transformers/05-gpt.md).

## O que vamos fazer aqui

O GPT gera texto da esquerda para a direita usando atenção causal, que impede cada posição de
olhar o futuro. Vamos implementar essa máscara do zero e confirmar que a matriz de atenção é
triangular inferior. Depois, um caminho opcional gera texto com um modelo de verdade via
Ollama. Este é o projeto que fecha o módulo.

## Atenção causal

A atenção causal é a atenção comum com uma máscara que zera as posições futuras antes da
softmax. O resultado é que cada posição só atende a si mesma e às anteriores.

In [ ]:
import numpy as np

def softmax(z, eixo=-1):
    z = z - z.max(axis=eixo, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=eixo, keepdims=True)


tokens = ["gato", "felino", "pulou", "alto"]
# Embeddings escolhidos: gato e felino são parecidos; pulou e alto, distintos.
E = np.array([
    [1.0, 0.0, 0.0, 0.0],   # gato
    [0.9, 0.1, 0.0, 0.0],   # felino (parecido com gato)
    [0.0, 0.0, 1.0, 0.0],   # pulou
    [0.0, 0.0, 0.0, 1.0],   # alto
])


def atencao_causal(X):
    """Atenção que impede cada posição de olhar para as futuras."""
    n, d = X.shape
    scores = X @ X.T / np.sqrt(d)
    mascara = np.triu(np.ones((n, n)), k=1).astype(bool)   # acima da diagonal
    scores[mascara] = -np.inf
    return softmax(scores, eixo=-1)


A = atencao_causal(E)
print("Matriz de atenção causal (triangular inferior):")
for i, t in enumerate(tokens):
    print(f"  {t:7} {np.round(A[i], 2)}")
print("\nNenhum peso vai para o futuro:", np.allclose(np.triu(A, 1), 0))

A matriz é triangular inferior, com a parte de cima zerada. Gato, na primeira
posição, só atende a si; felino atende a gato e a si; e assim por diante. Essa máscara simples
é o que separa um gerador de um leitor, e é a base de toda a família GPT.

## Caminho opcional, geração com um modelo de verdade

Esta célula usa o Ollama, localmente, para gerar uma continuação de texto de forma
autorregressiva, exatamente a ideia do GPT em escala. Roda apenas se o Ollama estiver
disponível.

In [ ]:
try:
    import ollama

    resposta = ollama.chat(
        model="llama3.1",
        messages=[{"role": "user", "content": (
            "Continue a frase em uma linha, de forma natural: "
            "Para estudar bem para uma prova, o aluno deve"
        )}],
    )
    print(resposta["message"]["content"])
except Exception as erro:
    print("Ollama não disponível agora. Veja docs/setup.md. Detalhe:", erro)

O modelo continua a frase prevendo uma palavra de cada vez, olhando só para trás, a
mesma ideia da atenção causal levada a uma escala enorme. Esse é o fim do arco que começou no
gerador de n-gramas do Módulo 1. Para o projeto do módulo, verifique as propriedades da sua
atenção (linhas somam 1, versão causal triangular) e conecte a atenção causal à geração dos
LLMs.